# Observational Evidence of IUCN threats across world map

Supplementary figure, not a manuscript finding on its own. One world-map choropleth per
threat, showing country-level unique-document counts for `Observational` negative-impact
records, using the same fixed-class binned scale as Result 02's country map.

In [1]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


In [2]:
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data_helpers").is_dir() and (candidate / "checklists").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the biodiversity repository root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [3]:
from data_helpers import results_config, visualization
from data_helpers.analysis.driver_composition import GEO_COUNT_BINS, GEO_COUNT_COLORS
from data_helpers.analysis.geo.count_plotting import add_class_colorbar, build_count_scale
from data_helpers.analysis.geo.geography import geo_counts, load_crosswalk, load_polygons
from data_helpers.labels import parse_list_labels
from data_helpers.prep import evidence_corpus_prep


## 1. Load the negative-impact evidence base

Eligible records with `s2_dir == "negative"` (a reported negative impact on
biodiversity), further restricted below to `Observational` study designs.


In [4]:
visualization.apply_style()

RESULTS_CONFIG = results_config.load_results_config()
FIGURES_CONFIG = RESULTS_CONFIG["threats_choropleth"]
EVIDENCE_CONFIG = RESULTS_CONFIG["biodiversity_evidence_prep"]
_, _, THREAT_DISPLAY_NAMES = results_config.threat_l0_display(RESULTS_CONFIG)

CORPUS_DIR = PROJECT_ROOT / EVIDENCE_CONFIG["output_directory"]
OUTPUT_DIR = PROJECT_ROOT / FIGURES_CONFIG["output_directory"]
TABLE_DIR = OUTPUT_DIR / FIGURES_CONFIG["table_subdirectory"]
FIGURE_DIR = OUTPUT_DIR / FIGURES_CONFIG["figure_subdirectory"]
for directory in (TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Provenance: the complete set of inputs this result depends on.
print("Inputs")
status = "found" if CORPUS_DIR.exists() else "MISSING"
print(f"  dataset  {CORPUS_DIR.relative_to(PROJECT_ROOT)}  [{status}]")
if not CORPUS_DIR.exists():
    raise FileNotFoundError("Run notebooks/data_processing/04_dataset.ipynb first.")
print(f"\nOutput section: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


Inputs
  dataset  notebooks/data_processing/outputs/04_dataset  [found]

Output section: notebooks/results/outputs/06_threats_choropleth


In [5]:
corpus_df = evidence_corpus_prep.BiodiversityEvidenceStore(CORPUS_DIR).load(
    columns=["UT", "s2_dir", "pred_threat_l0", "pred_countries", "pred_study_design"]
).publications

loss_df = corpus_df.loc[corpus_df["s2_dir"].eq("negative")].copy()
print(
    f"Of {len(corpus_df):,} eligible records, {len(loss_df):,} report a negative "
    "impact on biodiversity."
)


Of 605,199 eligible records, 253,081 report a negative impact on biodiversity.


## 2. Threat distribution across geography

One choropleth file per threat, excluding `Other Options` and `Unclear`. All files share
one count scale so they stay comparable to each other.


In [6]:
geo_source_df = loss_df.loc[
    loss_df["pred_study_design"].map(
        lambda value: tuple(parse_list_labels(value))
        == (FIGURES_CONFIG["geo_study_design"],)
    )
].copy()
print(
    f"Geography uses {geo_source_df['UT'].nunique():,} "
    f"{FIGURES_CONFIG['geo_study_design']} negative-impact records "
    f"(of {loss_df['UT'].nunique():,} total)."
)

geo_country_long, geo_country_audit = geo_counts(
    geo_source_df, level="country", by="pred_threat_l0", id_col="UT", verbose=False
)
print(geo_country_audit.summary())

crosswalk = load_crosswalk()
geo_country_long = (
    geo_country_long
    .merge(crosswalk[["iso3", "country"]], on="iso3", how="left")
    .rename(columns={"n": "record_count", "group_total": "threat_record_count"})
    .loc[:, [
        "pred_threat_l0", "iso3", "country",
        "record_count", "threat_record_count",
    ]]
    .sort_values(["pred_threat_l0", "record_count"], ascending=[True, False])
    .reset_index(drop=True)
)

geo_excluded_threats = set(FIGURES_CONFIG["geo_excluded_threats"])
geo_threat_plot_order = [
    threat
    for threat in (
        geo_country_long.groupby("pred_threat_l0", observed=True)["threat_record_count"]
        .first().sort_values(ascending=False).index.tolist()
    )
    if threat not in geo_excluded_threats
]
print(f"{len(geo_threat_plot_order)} threat panels: {geo_threat_plot_order}")


Geography uses 116,684 Observational negative-impact records (of 253,081 total).


country: 116,684 records; 0 with no geography ([] -> unclear)
    mapped         :  152,107 mentions
    not_applicable :   20,589 mentions
    unclear        :    2,611 mentions
    unresolved     :      949 mentions
    unresolved tokens: TAN (412), MAD (142), ROM (103), BUL (46), NEP (34), CAM (32), KOS (21), POR (18), DEN (18), CRO (14), BAN (13), SOL (7), locales [ (7), HON (7), URU (7)
10 threat panels: ['Pollution', 'Agriculture & Aquaculture', 'Climate Change & Severe Weather', 'Invasive & Other Problematic Species, Genes & Diseases', 'Natural System Modifications', 'Biological Resource Use', 'Residential & Commercial Development', 'Energy Production & Mining', 'Transportation & Service Corridors', 'Human Intrusions & Disturbance']


In [7]:
geo_country_long_path = TABLE_DIR / "geo_country_threat_long.csv"
geo_country_long.to_csv(geo_country_long_path, index=False)
print(geo_country_long_path.relative_to(PROJECT_ROOT))

print("Top 15 country x threat rows by document count")
display(geo_country_long.head(15))


notebooks/results/outputs/06_threats_choropleth/csv/geo_country_threat_long.csv
Top 15 country x threat rows by document count


,pred_threat_l0,iso3,country,record_count,threat_record_count
0,Agriculture & Aquaculture,CHN,China,2731,19225
1,Agriculture & Aquaculture,BRA,Brazil,1807,19225
2,Agriculture & Aquaculture,USA,United States of America (the),1531,19225
3,Agriculture & Aquaculture,IND,India,853,19225
4,Agriculture & Aquaculture,MEX,Mexico,634,19225
5,Agriculture & Aquaculture,ETH,Ethiopia,628,19225
6,Agriculture & Aquaculture,IDN,Indonesia,589,19225
7,Agriculture & Aquaculture,AUS,Australia,550,19225
8,Agriculture & Aquaculture,ESP,Spain,526,19225
9,Agriculture & Aquaculture,ARG,Argentina,485,19225


In [8]:
def _threat_slug(threat: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", threat.lower()).strip("_")


country_polygons = load_polygons(simplify_tolerance=0.2, preserve_topology=False)

geo_grid_counts = geo_country_long.loc[
    geo_country_long["pred_threat_l0"].isin(geo_threat_plot_order)
]
geo_scale_max = int(geo_grid_counts["record_count"].max())
geo_cmap, geo_norm, geo_bins = build_count_scale(
    geo_scale_max, bins=GEO_COUNT_BINS, colors=GEO_COUNT_COLORS
)


In [9]:
geo_threat_figure_paths = {}
for threat in geo_threat_plot_order:
    threat_counts = geo_country_long.loc[
        geo_country_long["pred_threat_l0"].eq(threat), ["iso3", "record_count"]
    ]
    merged = country_polygons.merge(
        threat_counts, left_on="ISO_3", right_on="iso3", how="left"
    )
    merged["record_count"] = merged["record_count"].fillna(0).astype(int)

    fig, ax = plt.subplots(figsize=(7.2, 3.5))
    merged.plot(
        ax=ax, color="#D9D9D9", edgecolor=visualization.BIODIVERSITY["neutral"],
        linewidth=0.15,
    )
    evidenced = merged.loc[merged["record_count"].gt(0)]
    if len(evidenced):
        evidenced.plot(
            ax=ax, column="record_count", cmap=geo_cmap, norm=geo_norm,
            edgecolor=visualization.BIODIVERSITY["neutral"], linewidth=0.15,
        )
    ax.set_xlim(-170, 180)
    ax.set_ylim(-58, 84)
    ax.set_axis_off()

    cax = fig.add_axes([0.28, 0.06, 0.44, 0.03])
    add_class_colorbar(fig, cax, geo_cmap, geo_norm, geo_bins, "Articles per country")
    fig.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.15)

    filename = f"{FIGURES_CONFIG['geo_choropleth_filename_prefix']}_{_threat_slug(threat)}"
    geo_threat_figure_paths[threat] = visualization.save_figure(
        fig, FIGURE_DIR / filename, formats=["pdf"]
    )
    plt.close(fig)

print(f"Saved {len(geo_threat_plot_order)} per-threat choropleths to {FIGURE_DIR.relative_to(PROJECT_ROOT)}")


✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_pollution.pdf
✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_agriculture_aquaculture.pdf


✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_climate_change_severe_weather.pdf
✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_invasive_other_problematic_species_genes_diseases.pdf


✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_natural_system_modifications.pdf
✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_biological_resource_use.pdf


✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_residential_commercial_development.pdf
✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_energy_production_mining.pdf


✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_transportation_service_corridors.pdf
✓ Saved: /Users/hunain/d/coding-projects/biodiversity/notebooks/results/outputs/06_threats_choropleth/figures/world_map_human_intrusions_disturbance.pdf
Saved 10 per-threat choropleths to notebooks/results/outputs/06_threats_choropleth/figures


## 3. Export


In [10]:
assert geo_country_long["record_count"].ge(0).all()
assert set(geo_threat_plot_order).isdisjoint(geo_excluded_threats)
assert geo_country_long["iso3"].str.len().eq(3).all()
assert len(geo_threat_figure_paths) == len(geo_threat_plot_order)


In [11]:
all_output_paths = [
    geo_country_long_path,
    *[path for paths in geo_threat_figure_paths.values() for path in paths],
]
for path in all_output_paths:
    print(path.relative_to(PROJECT_ROOT))

manifest = {
    "result": "06_threats_choropleth",
    "method_details": {
        "geography": {
            "study_design": FIGURES_CONFIG["geo_study_design"],
            "n_publications": int(geo_source_df["UT"].nunique()),
            "grain": "UT x country x threat",
            "metric": "unique-document count per country, not location quotient",
            "count_scale_bins": list(GEO_COUNT_BINS),
            "excluded_threats": sorted(geo_excluded_threats),
            "n_threat_files": int(len(geo_threat_plot_order)),
        },
    },
    "artifacts": {
        "csv/geo_country_threat_long.csv": (
            "Unique-document counts per country x threat."
        ),
        f"figures/{FIGURES_CONFIG['geo_choropleth_filename_prefix']}_<threat-slug>.pdf": (
            "One standalone country choropleth per threat, all sharing one count scale."
        ),
    },
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(manifest_path.relative_to(PROJECT_ROOT))


notebooks/results/outputs/06_threats_choropleth/csv/geo_country_threat_long.csv
notebooks/results/outputs/06_threats_choropleth/figures/world_map_pollution.pdf
notebooks/results/outputs/06_threats_choropleth/figures/world_map_agriculture_aquaculture.pdf
notebooks/results/outputs/06_threats_choropleth/figures/world_map_climate_change_severe_weather.pdf
notebooks/results/outputs/06_threats_choropleth/figures/world_map_invasive_other_problematic_species_genes_diseases.pdf
notebooks/results/outputs/06_threats_choropleth/figures/world_map_natural_system_modifications.pdf
notebooks/results/outputs/06_threats_choropleth/figures/world_map_biological_resource_use.pdf
notebooks/results/outputs/06_threats_choropleth/figures/world_map_residential_commercial_development.pdf
notebooks/results/outputs/06_threats_choropleth/figures/world_map_energy_production_mining.pdf
notebooks/results/outputs/06_threats_choropleth/figures/world_map_transportation_service_corridors.pdf
notebooks/results/outputs/06_t